# Importing ARCOS Data with Dask

Last week, we used dask to play with a few datasets to get a feel for how dask works. In order to help us develop code that would run quickly, however, we worked with very small, safe datasets. 

Today, we will continue to work with dask, but this time using much larger datasets. This means that (a) doing things incorrectly may lead to your computer crashing (So save all your open files before you start!), and (b) many of the commands you are being asked run will take several minutes each. 

For familiarity, and so you can see what advantages dask can bring to your workflow, today we'll be working with the DEA ARCOS drug shipment database published by the Washington Post! However, to strike a balance between size and speed, we'll be working with a slightly thinned version that has only the last two years of data, instead of all six.

## Exercise 1

Download the thinned ARCOS data [from this link](https://www.dropbox.com/s/o7nc6yvrwog4ozi/arcos_2011_2012.tsv.zip?dl=0). It should be about 2GB zipped, 25 GB unzipped. 

## Exercise 2

Our goal today is going to be to find the pharmaceutical company that has shipped the most opioids (`MME_Conversion_Factor * CALC_BASE_WT_IN_GM`) in the US.

When working with large datasets, it is good practice to begin by prototyping your code with a subset of your data. So begin by using `pandas` to read in the first 100,000 lines of the ARCOS data and write pandas code to compute the shipments from each shipper (the group that reported the shipment). 

In [1]:
import pandas as pd
import zipfile

zip_path = "/Users/brucechen/Documents/IDS720 PDS/arcos_2011_2012.tsv.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    # List files inside the ZIP
    print("Files in zip:", z.namelist())

    # Extract only the TSV file (ignore __MACOSX files)
    tsv_files = [f for f in z.namelist() if f.endswith(".tsv")]
    if not tsv_files:
        raise FileNotFoundError("No TSV file found in zip file.")

    # Assuming we want the first TSV file found
    tsv_name = tsv_files[0]
    print(f"Reading {tsv_name} ...")

    with z.open(tsv_name) as f:
        df = pd.read_csv(f, sep="\t", nrows=100_000, low_memory=False)
        pd.set_option("mode.copy_on_write", True)

Files in zip: ['arcos_2011_2012.tsv', '__MACOSX/', '__MACOSX/._arcos_2011_2012.tsv']
Reading arcos_2011_2012.tsv ...


In [2]:
df.head()

,Unnamed: 0,REPORTER_DEA_NO,REPORTER_BUS_ACT,REPORTER_NAME,REPORTER_ADDL_CO_INFO,REPORTER_ADDRESS1,REPORTER_ADDRESS2,REPORTER_CITY,REPORTER_STATE,REPORTER_ZIP,...,Product_Name,Ingredient_Name,Measure,MME_Conversion_Factor,Combined_Labeler_Name,Revised_Company_Name,Reporter_family,dos_str,date,year
0,0,PA0006836,DISTRIBUTOR,ACE SURGICAL SUPPLY CO INC,NaN,1034 PEARL STREET,NaN,BROCKTON,MA,2301,...,HYDROCODONE BIT/ACETA 10MG/500MG USP,HYDROCODONE BITARTRATE HEMIPENTAHYDRATE,TAB,1.0,SpecGx LLC,Mallinckrodt,ACE Surgical Supply Co Inc,10.0,2012-12-26,2012
1,9,PA0021179,DISTRIBUTOR,APOTHECA INC,NaN,1622 N 16TH ST,NaN,PHOENIX,AZ,85006,...,HYDROCODONE BITARTRATE & ACETA 5MG/,HYDROCODONE BITARTRATE HEMIPENTAHYDRATE,TAB,1.0,Apotheca Inc.,Apotheca Inc.,Apotheca Inc,5.0,2012-12-05,2012
2,10,PA0021179,DISTRIBUTOR,APOTHECA INC,NaN,1622 N 16TH ST,NaN,PHOENIX,AZ,85006,...,HYDROCODONE BITARTRATE & ACETA 5MG/,HYDROCODONE BITARTRATE HEMIPENTAHYDRATE,TAB,1.0,Apotheca Inc.,Apotheca Inc.,Apotheca Inc,5.0,2012-07-24,2012
3,16,PA0021179,DISTRIBUTOR,APOTHECA INC,NaN,1622 N 16TH ST,NaN,PHOENIX,AZ,85006,...,HYDROCODONEBITARTRATE & ACETA 7.5MG,HYDROCODONE BITARTRATE HEMIPENTAHYDRATE,TAB,1.0,Apotheca Inc.,Apotheca Inc.,Apotheca Inc,7.5,2012-02-04,2012
4,17,PA0021179,DISTRIBUTOR,APOTHECA INC,NaN,1622 N 16TH ST,NaN,PHOENIX,AZ,85006,...,HYDROCODONE BITARTRATE & ACETA 5MG/,HYDROCODONE BITARTRATE HEMIPENTAHYDRATE,TAB,1.0,Apotheca Inc.,Apotheca Inc.,Apotheca Inc,5.0,2011-11-07,2011


In [3]:
df.columns

Index(['Unnamed: 0', 'REPORTER_DEA_NO', 'REPORTER_BUS_ACT', 'REPORTER_NAME',
       'REPORTER_ADDL_CO_INFO', 'REPORTER_ADDRESS1', 'REPORTER_ADDRESS2',
       'REPORTER_CITY', 'REPORTER_STATE', 'REPORTER_ZIP', 'REPORTER_COUNTY',
       'BUYER_DEA_NO', 'BUYER_BUS_ACT', 'BUYER_NAME', 'BUYER_ADDL_CO_INFO',
       'BUYER_ADDRESS1', 'BUYER_ADDRESS2', 'BUYER_CITY', 'BUYER_STATE',
       'BUYER_ZIP', 'BUYER_COUNTY', 'TRANSACTION_CODE', 'DRUG_CODE', 'NDC_NO',
       'DRUG_NAME', 'QUANTITY', 'UNIT', 'ACTION_INDICATOR', 'ORDER_FORM_NO',
       'CORRECTION_NO', 'STRENGTH', 'TRANSACTION_DATE', 'CALC_BASE_WT_IN_GM',
       'DOSAGE_UNIT', 'TRANSACTION_ID', 'Product_Name', 'Ingredient_Name',
       'Measure', 'MME_Conversion_Factor', 'Combined_Labeler_Name',
       'Revised_Company_Name', 'Reporter_family', 'dos_str', 'date', 'year'],
      dtype='object')

In [4]:
df["total_mme"] = df["MME_Conversion_Factor"] * df["CALC_BASE_WT_IN_GM"]

shipment_by_shipper = (
    df.groupby("REPORTER_NAME")["total_mme"]
    .sum()
    .reset_index()
    .sort_values(by="total_mme", ascending=False)
)

shipment_by_shipper

,REPORTER_NAME,total_mme
20,MCKESSON CORPORATION,299266.331225
7,"CARDINAL HEALTH 110, LLC",54352.323711
2,AMERISOURCEBERGEN DRUG CORP,34561.394892
17,KINRAY INC,28620.315246
19,LOUISIANA WHOLESALE DRUG CO,14787.765559
11,FRANK W KERR INC,8730.016283
12,H D SMITH WHOLESALE DRUG CO,6399.324050
16,KAISER FOUNDATION HOSPITALS,3891.329580
5,BURLINGTON DRUG COMPANY,3889.490325
1,AMERICAN SALES COMPANY,3432.058005


## Exercise 3

Now let's turn to dask. Re-write your code for dask, and calculate the total shipments by reporting company. Remember: 

- Activate a conda environment with a clean dask installation.
- Start by spinning up a distributed cluster.
- Dask won't read compressed files, so you have to unzip your ARCOS data. 
- Start your cluster in a cell all by itself since you don't want to keep re-running the "start a cluster" code. 

If you need to review dask basic code, [check here](https://nickeubank.github.io/practicaldatascience_book/notebooks/PDS_not_yet_in_coursera/30_big_data/70_dask.html).

As you run your code, make sure to click on the Dashboard link below where you created your cluster:

![dask_dashboard](images/dask_cluster.png)

Among other things, the bar across the bottom should give you a sense of how long your task will take:

![dask_progress](images/dask_progress.png)

(For context, my computer (which has 10 cores) only took a couple seconds. My computer is fast, but most computers should be done within a couple minutes, tops).


In [ ]:
from dask.distributed import Client, LocalCluster

cluster = LocalCluster()
client = Client(cluster)

client

/Users/brucechen/miniforge3/lib/python3.12/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 56975 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:56975/status,
Dashboard: http://127.0.0.1:56975/status,Workers: 4
Total threads: 8,Total memory: 24.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:56976,Workers: 0
Dashboard: http://127.0.0.1:56975/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:56989,Total threads: 2
Dashboard: http://127.0.0.1:56994/status,Memory: 6.00 GiB
Nanny: tcp://127.0.0.1:56979,


2025-11-20 11:01:22,075 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 344e3d438b78faa17bc8bb8dd6b44c05 initialized by task ('shuffle-transfer-344e3d438b78faa17bc8bb8dd6b44c05', 231) executed on worker tcp://127.0.0.1:56987
2025-11-20 11:01:43,305 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 344e3d438b78faa17bc8bb8dd6b44c05 deactivated due to stimulus 'task-finished-1763654503.303464'
2025-11-20 11:08:10,980 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 04a1f57a132a80aa262767882a44be6c initialized by task ('shuffle-transfer-04a1f57a132a80aa262767882a44be6c', 99) executed on worker tcp://127.0.0.1:56989
2025-11-20 11:08:37,566 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 04a1f57a132a80aa262767882a44be6c deactivated due to stimulus 'task-finished-1763654917.564619'
2025-11-20 11:11:16,040 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 1ade9704988b3fa0d71ab271a5ed2fa6 initialized by task ('shuffle-transfer-1ade9704988b3

In [30]:
import dask.dataframe as dd

tsv_path = "/Users/brucechen/Documents/IDS720 PDS/arcos_2011_2012.tsv"

cols = {
    "CALC_BASE_WT_IN_GM": "float64",
    "MME_Conversion_Factor": "float64",
    "REPORTER_NAME": "object",
    "REPORTER_STATE": "object",
    "REPORTER_COUNTY": "object",
    "year": "int64",
}

ddf = dd.read_csv(
    tsv_path, sep="\t", assume_missing=True, dtype=cols, usecols=list(cols.keys())
)

In [33]:
ddf.head()

,REPORTER_NAME,REPORTER_STATE,REPORTER_COUNTY,CALC_BASE_WT_IN_GM,MME_Conversion_Factor,year
0,ACE SURGICAL SUPPLY CO INC,MA,PLYMOUTH,0.60540,1.0,2012
1,APOTHECA INC,AZ,MARICOPA,0.45405,1.0,2012
2,APOTHECA INC,AZ,MARICOPA,0.60540,1.0,2012
3,APOTHECA INC,AZ,MARICOPA,1.36215,1.0,2012
4,APOTHECA INC,AZ,MARICOPA,0.60540,1.0,2011


In [35]:
ddf["total_mme"] = ddf["MME_Conversion_Factor"] * ddf["CALC_BASE_WT_IN_GM"]

shipment = ddf.groupby("REPORTER_NAME")["total_mme"].sum().reset_index()

result = shipment.compute()
result = result.sort_values(by="total_mme", ascending=False)

result.head(15)

,REPORTER_NAME,total_mme
2,MCKESSON CORPORATION,5.604679e+07
5,CARDINAL HEALTH,4.671958e+07
3,WALGREEN CO,4.185033e+07
7,AMERISOURCEBERGEN DRUG CORP,2.553364e+07
8,"CARDINAL HEALTH 110, LLC",5.896801e+06
4,SMITH DRUG COMPANY,5.529262e+06
8,WAL-MART PHARMACY WHSE #45,5.135986e+06
1,MORRIS & DICKSON CO,2.825622e+06
8,CVS INDIANA,2.624450e+06
1,"CVS TN DISTRIBUTION, LLC",2.505745e+06


## Exercise 4

Now let's calculate, *for each state*, what company shipped the most pills?

Note you will quickly find that you can't sort in dask -- sorting in parallel is *really* tricky! So you'll have to work around that. Do what you need to do on the big dataset first, then compute it all so you get it as a regular pandas dataframe, then finish. 

Does this seem like a situation where a single company is responsible for the opioid epidemic?

In [27]:
by_state = (
    ddf.groupby(["REPORTER_STATE", "REPORTER_NAME"])["total_mme"].sum().reset_index()
)

result_state = by_state.compute()
result_state = result_state.sort_values(by="total_mme", ascending=False)

result_state.head(15)

,REPORTER_STATE,REPORTER_NAME,total_mme
7,OH,WALGREEN CO,1.678570e+07
1,CA,WALGREEN CO,1.019160e+07
2,FL,WALGREEN CO,9.158992e+06
0,NY,"CARDINAL HEALTH 110, LLC",5.896801e+06
0,CA,MCKESSON CORPORATION,5.433826e+06
5,WV,CARDINAL HEALTH,5.402427e+06
2,NJ,MCKESSON CORPORATION,5.143832e+06
10,AR,WAL-MART PHARMACY WHSE #45,5.135986e+06
11,CA,AMERISOURCEBERGEN DRUG CORP,4.789604e+06
1,FL,MCKESSON CORPORATION,4.523626e+06


**It seems a like more than one company - namely, Walgreen, Cardinal Health, McKesson Corp - are all shipping a lot of pills.**

## Exercise 5 

Now go ahead and try and re-do the chunking you did by hand for your project (with this 2 years of data) -- calculate, for each year, the total morphine equivalents sent to each county in the US. 

In [36]:
by_county_year = (
    ddf.groupby(["year", "REPORTER_COUNTY"])["total_mme"].sum().reset_index()
)

result_county_year = by_county_year.compute()
result_county_year = result_county_year.sort_values(
    by=["year", "total_mme"], ascending=[True, False]
)

result_county_year.head(15)

,year,REPORTER_COUNTY,total_mme
8,2011,WOOD,7.950804e+06
14,2011,YOLO,6.236477e+06
14,2011,PALM BEACH,6.074696e+06
0,2011,POLK,5.834193e+06
19,2011,BENTON,3.461055e+06
0,2011,MARICOPA,3.403965e+06
11,2011,ORANGE,3.395410e+06
0,2011,LOS ANGELES,3.328478e+06
13,2011,KNOX,3.112345e+06
3,2011,GLOUCESTER,3.060927e+06


## Exercise 6

Now, re-write your opioid project's initial opioid import using dask. Each person on your team should create a NEW branch to try this. The person who wrote the initial chunking code can help everyone else understand what they did originally and the data, but everyone should write their own code. 

**WARNING:** You will probably run into a lot of type errors (depending on how the ARCOS data has changed since last year). With real world messy data one of the biggest problems with dask is that it struggles if halfway through dataset it discovers that the column it *thought* was floats contains text. That's why, in the dask reading, [I specified the column type for so many columns](https://nickeubank.github.io/practicaldatascience_book/notebooks/PDS_not_yet_in_coursera/30_big_data/70_dask.html#what-can-dask-do-for-me) as `objects` explicitly. Then, because occasionally there data cleanliness issues, I had to do some converting data types by hand. 